Se utiliza:
    {'n_estimators': 5, 'tree_max_depth': 3, 'c': 0.25},

# 1. Import and load

In [1]:
import os
from pathlib import Path
import sys
import pickle as pk
import numpy as np
import pandas as pd

project_root = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, 'src', 'model'))
sys.path.append(os.path.join(project_root, 'src', 'rules'))
sys.path.append(os.path.join(project_root, 'src', 'fuzzy'))

from rulecosi import RuleCOSIClassifier
from src.model import train_model as mod
from config import LOAN_FILE, MODEL_DIR, PKL_DIR
import glob

# Cargar la porción COSI (datos que el modelo base nunca vio)
X_train_cosi = pd.read_pickle(os.path.join(PKL_DIR, 'X_train_cosi.pkl'))
y_train_cosi = pd.read_pickle(os.path.join(PKL_DIR, 'y_train_cosi.pkl'))

X_test = pd.read_pickle(os.path.join(PKL_DIR, 'X_test.pkl'))
y_test = pd.read_pickle(os.path.join(PKL_DIR, 'y_test.pkl'))


In [2]:
# Carga del modelo base CatBoost
pklList = mod.read_all_pkl_files(PKL_DIR)
prefix  = 'catboost_DANI_encoded_'

try:
    search_pattern = os.path.join(MODEL_DIR, f'{prefix}*.cbm')
    model_files    = glob.glob(search_pattern)
    if not model_files:
        raise FileNotFoundError(f"No se encontraron modelos con el prefijo '{prefix}' en: {MODEL_DIR}")
    latest_model_path = sorted(model_files)[-1]
    print(f'[INFO] Modelo más reciente detectado: {os.path.basename(latest_model_path)}')
    catboost_model, catboost_metrics, X_train_encoded_base, cat_mappings = mod.load_model(latest_model_path)
    print(f'[SUCCESS] Modelo cargado:')
    print(f'  - Model type     : {catboost_metrics.get("model_type")}')
    print(f'  - Saved date     : {catboost_metrics.get("saved_date")}')
    print(f'  - Opt. threshold : {catboost_metrics.get("optimal_threshold")}')
except Exception as e:
    print(f'[ERROR] No se pudo cargar el modelo: {e}')


[INFO] Modelo más reciente detectado: catboost_DANI_encoded_20260525_091324_model.cbm
[INFO] CatBoost model loaded from: c:\Users\danli\OneDrive\Casa Serrano Hidalgo\Danilo\Maestria USFQ Danilo\Tesis\Rule reduction\Tesis Byron\COSI_Aplied\01 FUZZY INFERENCE SYSTEM\object\model\catboost_DANI_encoded_20260525_091324_model.cbm
[INFO] Metrics loaded from: c:\Users\danli\OneDrive\Casa Serrano Hidalgo\Danilo\Maestria USFQ Danilo\Tesis\Rule reduction\Tesis Byron\COSI_Aplied\01 FUZZY INFERENCE SYSTEM\object\model\catboost_DANI_encoded_20260525_091324_metrics.json
[INFO] Category mappings loaded from: c:\Users\danli\OneDrive\Casa Serrano Hidalgo\Danilo\Maestria USFQ Danilo\Tesis\Rule reduction\Tesis Byron\COSI_Aplied\01 FUZZY INFERENCE SYSTEM\object\model\catboost_DANI_encoded_20260525_091324_category_mappings.pkl
  Encoded features: ['debt_settlement_flag', 'grade', 'verification_status']
[INFO] X_train_encoded loaded from: c:\Users\danli\OneDrive\Casa Serrano Hidalgo\Danilo\Maestria USFQ Dani

## 1.1 Métricas del modelo seleccionado: Catboost

In [3]:
catboost_metrics


{'model_type': 'CatBoost',
 'categorical_features_encoded': ['debt_settlement_flag',
  'grade',
  'verification_status'],
 'threshold_range': [0.4, 0.65],
 'train_recall': 0.8476773314951496,
 'train_precision': 0.9405535484432616,
 'train_f1': 0.8917035613434182,
 'train_auc': 0.9818222464157803,
 'train_logloss': 0.10730479119130527,
 'val_recall': 0.8428603452140776,
 'val_precision': 0.9350035746479749,
 'val_f1': 0.8865441577388409,
 'val_auc': 0.9803170533445922,
 'val_logloss': 0.11156492184818943,
 'confusion_matrix': array([[725124,   9560],
        [ 27180, 151257]]),
 'val_confusion_matrix': array([[144846,   2091],
        [  5608,  30080]]),
 'best_iteration': 115,
 'optimal_threshold': 0.4800000000000001,
 'classification_threshold': 0.4800000000000001,
 'model_name': 'catboost_DANI_encoded',
 'saved_timestamp': '20260525_091324',
 'saved_date': '2026-05-25 09:13:24',
 'has_category_mappings': True,
 'encoded_features': ['debt_settlement_flag', 'grade', 'verification_stat

In [4]:
n_estimators = catboost_model.tree_count_
print('# de árboles:', n_estimators)
cm = catboost_metrics['val_confusion_matrix']
support_0, support_1 = cm[0].sum(), cm[1].sum()
total_support = support_0 + support_1
prec_0 = cm[0,0] / (cm[0,0] + cm[1,0])
rec_0  = cm[0,0] / (cm[0,0] + cm[0,1])
f1_0   = 2 * (prec_0 * rec_0) / (prec_0 + rec_0)
prec_1, rec_1, f1_1 = catboost_metrics['val_precision'], catboost_metrics['val_recall'], catboost_metrics['val_f1']
accuracy     = (cm[0,0] + cm[1,1]) / total_support
macro_avg    = [np.mean([prec_0, prec_1]), np.mean([rec_0, rec_1]), np.mean([f1_0, f1_1])]
weighted_avg = [
    (prec_0 * support_0 + prec_1 * support_1) / total_support,
    (rec_0  * support_0 + rec_1  * support_1) / total_support,
    (f1_0   * support_0 + f1_1   * support_1) / total_support
]
print(f"{'':>15} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
print('')
print(f"{'0':>15} {prec_0:>10.2f} {rec_0:>10.2f} {f1_0:>10.2f} {support_0:>10}")
print(f"{'1':>15} {prec_1:>10.2f} {rec_1:>10.2f} {f1_1:>10.2f} {support_1:>10}")
print('')
print(f"{'accuracy':>15} {'':>10} {'':>10} {accuracy:>10.2f} {total_support:>10}")
print(f"{'macro avg':>15} {macro_avg[0]:>10.2f} {macro_avg[1]:>10.2f} {macro_avg[2]:>10.2f} {total_support:>10}")
print(f"{'weighted avg':>15} {weighted_avg[0]:>10.2f} {weighted_avg[1]:>10.2f} {weighted_avg[2]:>10.2f} {total_support:>10}")


# de árboles: 116
                 precision     recall   f1-score    support

              0       0.96       0.99       0.97     146937
              1       0.94       0.84       0.89      35688

       accuracy                             0.96     182625
      macro avg       0.95       0.91       0.93     182625
   weighted avg       0.96       0.96       0.96     182625


## 1.2 Encoding de X_train_cosi y X_test

In [5]:
# Las columnas categóricas se codifican usando los mismos mappings con los que
# fue entrenado CatBoost (label encoding: A->0, B->1, etc.).
# Esto garantiza consistencia entre el espacio de datos que vio el modelo base
# y el que recibe RuleCOSI+ - sin fuga de informacion del target.

X_train_encoded = mod.encode_data_like_training(X_train_cosi, cat_mappings)
X_test_encoded  = mod.encode_data_like_training(X_test, cat_mappings)

print('Encoding aplicado a X_train_cosi y X_test usando cat_mappings del modelo base.')
print(f'  X_train_encoded shape: {X_train_encoded.shape}')
print(f'  X_test_encoded  shape: {X_test_encoded.shape}')
print(f'  Columnas codificadas : {list(cat_mappings.keys())}')


Encoding aplicado a X_train_cosi y X_test usando cat_mappings del modelo base.
  X_train_encoded shape: (391338, 12)
  X_test_encoded  shape: (559054, 12)
  Columnas codificadas : ['debt_settlement_flag', 'grade', 'verification_status']


# 2. Rule Cosi+ implementation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

param_grid = [
    # n_estimadores = 2, tree_max_depth = 2, c = 0.85, 0.90, 0.95

    {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.30},
    {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.35},
    {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.40},
    {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.45},
    {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.50},
  

]

skf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

print('Iniciando busqueda con CV (5 Folds por combinacion)...')
print('='*70)

for p_idx, params in enumerate(param_grid):
    print(f'\n[Configuracion {p_idx+1}/{len(param_grid)}]: {params}')
    print('-'*70)
    fold_f1s = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_encoded, y_train_cosi)):
        X_f_train = X_train_encoded.iloc[train_idx]
        y_f_train = y_train_cosi.iloc[train_idx]
        X_f_val   = X_train_encoded.iloc[val_idx]
        y_f_val   = y_train_cosi.iloc[val_idx]

        rc = RuleCOSIClassifier(base_ensemble=catboost_model, **params, random_state=42)
        rc.fit(X_f_train, y_f_train)

        _, _, val_f1, _ = precision_recall_fscore_support(
            y_f_val, rc.predict(X_f_val), average='binary', zero_division=0)
        fold_f1s.append(val_f1)
        print(f' Fold {fold+1} finalizado. Val F1: {val_f1:.4f}')

    mean_f1 = sum(fold_f1s) / len(fold_f1s)

    rc_temp = RuleCOSIClassifier(base_ensemble=catboost_model, **params, random_state=42)
    rc_temp.fit(X_train_encoded, y_train_cosi)
    n_rules = len(rc_temp.simplified_ruleset_.rules)

    all_results.append({
        'params_dict': params,
        'params_str':  str(params),
        'val_f1':      round(mean_f1, 6),
        'n_rules':     n_rules,
    })
    print(f'  -> Promedio Val F1: {mean_f1:.4f} | Reglas: {n_rules}')

df_results = pd.DataFrame(all_results).sort_values('n_rules').reset_index(drop=True)

print('\n' + '='*70)
print(' TABLA PARETO - Val F1 vs. # Reglas (ordenada por # reglas)')
print('='*70)
print(f"{'#':>3}  {'Params':<50}  {'Val F1':>8}  {'Reglas':>7}")
print('-'*75)
for i, row in df_results.iterrows():
    print(f"{i+1:>3}  {row['params_str']:<50}  {row['val_f1']:>8.4f}  {row['n_rules']:>7}")

f1_catboost = catboost_metrics['val_f1']
umbral      = f1_catboost * 0.85

fig, ax = plt.subplots(figsize=(10, 6))
colores = ['green' if r['val_f1'] >= umbral else 'red' for _, r in df_results.iterrows()]
ax.scatter(df_results['n_rules'], df_results['val_f1'], c=colores, s=90, zorder=3)

for i, row in df_results.iterrows():
    lbl = f"d={row['params_dict']['tree_max_depth']},c={row['params_dict']['c']},n={row['params_dict']['n_estimators']}"
    ax.annotate(lbl, (row['n_rules'], row['val_f1']),
                textcoords='offset points', xytext=(5, 4), fontsize=7)

ax.axhline(y=umbral,      color='orange', linestyle='--', label=f'Umbral 85% ({umbral:.4f})')
ax.axhline(y=f1_catboost, color='blue',   linestyle=':',  label=f'CatBoost ({f1_catboost:.4f})')
ax.axvline(x=40,          color='gray',   linestyle='-.', alpha=0.5, label='Target: 40 reglas')

from matplotlib.lines import Line2D
extra = [Line2D([0],[0],marker='o',color='w',markerfacecolor='green',ms=8,label='F1 >= 85% CB'),
         Line2D([0],[0],marker='o',color='w',markerfacecolor='red',  ms=8,label='F1 < 85% CB')]
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles+extra, fontsize=9)
ax.set_xlabel('Numero de Reglas')
ax.set_ylabel('Val F1 (CV promedio)')
ax.set_title('Frontera Pareto: Complejidad vs. Rendimiento')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pareto_cv.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafico guardado como pareto_cv.png')


Iniciando busqueda con CV (5 Folds por combinacion)...

[Configuracion 1/5]: {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.3}
----------------------------------------------------------------------
 Fold 1 finalizado. Val F1: 0.8793
 Fold 2 finalizado. Val F1: 0.8822
 Fold 3 finalizado. Val F1: 0.8833
 Fold 4 finalizado. Val F1: 0.8817
 Fold 5 finalizado. Val F1: 0.8790
  -> Promedio Val F1: 0.8811 | Reglas: 136

[Configuracion 2/5]: {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.35}
----------------------------------------------------------------------
 Fold 1 finalizado. Val F1: 0.8805
 Fold 2 finalizado. Val F1: 0.8823
 Fold 3 finalizado. Val F1: 0.8825
 Fold 4 finalizado. Val F1: 0.8819
 Fold 5 finalizado. Val F1: 0.8789
  -> Promedio Val F1: 0.8812 | Reglas: 87

[Configuracion 3/5]: {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.4}
----------------------------------------------------------------------
 Fold 1 finalizado. Val F1: 0.8773
 Fold 2 finalizado. Val F1: 0.8817
 Fold 3

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SELECCION MANUAL DEL GANADOR
# Mira la tabla y el grafico Pareto de arriba y escribe aqui
# los parametros del punto que mejor equilibra F1 y reglas.
# ══════════════════════════════════════════════════════════════════════
best_config = {
    'n_estimators':   2,     # <── cambia aqui
    'tree_max_depth': 2,     # <── cambia aqui
    'c':              0.35,  # <── cambia aqui
}

print('Config seleccionada:', best_config)

# Entrenamiento final con la totalidad de X_train_cosi
print('\nEntrenando modelo final con la totalidad de X_train_cosi...')
final_rc = RuleCOSIClassifier(base_ensemble=catboost_model, **best_config, random_state=42)
final_rc.fit(X_train_encoded, y_train_cosi)
print(f'Modelo final listo. Reglas extraidas: {len(final_rc.simplified_ruleset_.rules)}')


Config seleccionada: {'n_estimators': 2, 'tree_max_depth': 2, 'c': 0.95}

Entrenando modelo final con la totalidad de X_train_cosi...
Modelo final listo. Reglas extraidas: 88


# 3. Evaluacion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
import pandas as pd
import numpy as np

# --- 1. EVALUACION DE CATBOOST SOBRE X_TEST (mismo conjunto que RuleCOSI+) ---
X_test_encoded_cb = mod.encode_data_like_training(X_test, cat_mappings)
threshold  = catboost_metrics['optimal_threshold']
y_pred_cb  = (catboost_model.predict_proba(X_test_encoded_cb)[:, 1] >= threshold).astype(int)

prec_1_cb, rec_1_cb, f1_1_cb, _ = precision_recall_fscore_support(y_test, y_pred_cb, average='binary')

print('\n' + '='*60)
print('CATBOOST - EVALUACION SOBRE X_TEST')
print('='*60)
print(classification_report(y_test, y_pred_cb, digits=4))

# --- 2. EVALUACION DE RULE-COSI+ SOBRE X_TEST ---
y_pred_rc = final_rc.predict(X_test_encoded)
cm_rc     = confusion_matrix(y_test, y_pred_rc)
prec_1_rc, rec_1_rc, f1_1_rc, _ = precision_recall_fscore_support(y_test, y_pred_rc, average='binary')

print('\n' + '='*60)
print('RULE-COSI+ - EVALUACION SOBRE X_TEST')
print('='*60)
print(classification_report(y_test, y_pred_rc, digits=4))

# --- 3. TABLA COMPARATIVA (ambos evaluados en el mismo X_test) ---
comparativa_tesis = pd.DataFrame({
    'Modelo':                  ['CatBoost', 'Rule-COSI+'],
    'F1-Score (Clase 1)':      [f1_1_cb,   f1_1_rc],
    'Precision (Clase 1)':     [prec_1_cb, prec_1_rc],
    'Recall (Clase 1)':        [rec_1_cb,  rec_1_rc],
    'Complejidad (#unidades)': [catboost_model.tree_count_, len(final_rc.simplified_ruleset_.rules)],
    'Tipo de Unidad':          ['Arboles', 'Reglas']
})

print('\n' + '='*60)
print('RESULTADOS FINALES PARA LA TESIS')
print('(ambos modelos evaluados sobre el mismo X_test)')
print('='*60)
print(comparativa_tesis.to_string(index=False))

# --- 4. GRAFICOS ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

df_plot = comparativa_tesis.melt(id_vars='Modelo', value_vars=['F1-Score (Clase 1)', 'Precision (Clase 1)', 'Recall (Clase 1)'])
sns.barplot(data=df_plot, x='variable', y='value', hue='Modelo', palette='viridis', ax=ax1)
ax1.set_title('Rendimiento: CatBoost vs Rule-COSI+', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score')
ax1.set_xlabel('')
ax1.set_ylim(0, 1.1)
ax1.legend(loc='lower right')

sns.barplot(x='Modelo', y='Complejidad (#unidades)', data=comparativa_tesis, palette='magma', ax=ax2)
for i, val in enumerate(comparativa_tesis['Complejidad (#unidades)']):
    ax2.text(i, val + 1, f'{int(val)} {comparativa_tesis["Tipo de Unidad"][i]}', ha='center', fontweight='bold', fontsize=12)
ax2.set_title('Reduccion de Complejidad (Interpretabilidad)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Cantidad (Arboles vs Reglas)')

plt.tight_layout()
plt.show()

# --- 5. REGLAS FINALES ---
print('\n' + '='*60)
print('REGLAS FINALES EXTRAIDAS (CONOCIMIENTO DESCUBIERTO)')
print('='*60)
for i, rule in enumerate(final_rc.simplified_ruleset_.rules):
    print(f'Regla {i+1}: {rule}')



CATBOOST - EVALUACION SOBRE X_TEST
              precision    recall  f1-score   support

           0     0.9634    0.9864    0.9747    450222
           1     0.9375    0.8448    0.8887    108832

    accuracy                         0.9588    559054
   macro avg     0.9504    0.9156    0.9317    559054
weighted avg     0.9583    0.9588    0.9580    559054


RULE-COSI+ - EVALUACION SOBRE X_TEST
              precision    recall  f1-score   support

           0     0.9671    0.9779    0.9725    450222
           1     0.9042    0.8622    0.8827    108832

    accuracy                         0.9554    559054
   macro avg     0.9356    0.9201    0.9276    559054
weighted avg     0.9548    0.9554    0.9550    559054


RESULTADOS FINALES PARA LA TESIS
(ambos modelos evaluados sobre el mismo X_test)
    Modelo  F1-Score (Clase 1)  Precision (Clase 1)  Recall (Clase 1)  Complejidad (#unidades) Tipo de Unidad
  CatBoost            0.888697             0.937453          0.844761           